# Multi-Country Comparison

This recipe compares EVI seasonality across neighboring countries. The quickstart analyzes a single country; here we loop over three East African countries to show cross-border patterns.

This recipe demonstrates `plot_time_series_by_region` (faceted time-series charts) and `plot_choropleth` (map colored by EVI values), which are in the public API but not covered in the quickstart.

In [ ]:
# papermill parameters
countries = ["RWA", "BDI", "UGA"]
admin_level = 1
start_date = "2023-01-01"
end_date = "2023-12-31"
quick_mode = False


In [ ]:
if quick_mode:
    countries = countries[:1]
    end_date = "2023-03-31"


In [ ]:
import evy
import pandas as pd

## Load boundaries for three countries

In [ ]:
gdfs = {iso: evy.get_boundaries(iso, admin_level=admin_level) for iso in countries}

# Add country column for later grouping
for iso, gdf in gdfs.items():
    gdf["country"] = iso

## Compute monthly EVI for each country

Each country is processed independently — evy does not mosaic across borders. We concatenate the results into a single DataFrame for visualization.

In [ ]:
dfs = []
for iso, gdf in gdfs.items():
    df = evy.zonal_stats(
        gdf,
        zone_col="shapeName",
        start_date=start_date,
        end_date=end_date,
        freq=evy.MONTHLY,
        stats=["mean"],
    )
    df["country"] = iso
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all.head()

## Faceted time series by region

`plot_time_series_by_region` creates a small-multiples chart — one panel per region. Here we show the regions from one country:

In [ ]:
df_rwa = df_all[df_all["country"] == "RWA"]
evy.plot_time_series_by_region(df_rwa, region_col="shapeName")

## Choropleth map

`plot_choropleth` colors each administrative unit by its mean EVI for a given period. We compute the annual mean across all months and map it.

**Note:** `plot_choropleth` joins the data DataFrame to the GeoDataFrame using a shared key column. GeoBoundaries data uses `shapeName` for region names — we pass that as both `geo_key` and `df_key`.

In [ ]:
# Annual mean per region for Rwanda
df_rwa_annual = (
    df_rwa.groupby("shapeName", as_index=False)["mean"].mean()
)

evy.plot_choropleth(
    df_rwa_annual,
    gdfs["RWA"],
    value_col="mean",
    region_col="shapeName",
    geo_key="shapeName",
    df_key="shapeName",
    title="Mean EVI by Province — Rwanda 2023",
)

Each country is analyzed independently — evy does not stitch rasters or boundaries across borders. This is intentional: cross-border mosaicking introduces CRS alignment issues and complicates the methodology section of your paper. Compare the countries' results by placing their charts side by side, as shown above.